In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_x = train.drop(columns=['efs'])
train_y = train['efs']

for i in test:
    test[i]=test[i].to_numpy()
    
#train.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

clf = GradientBoostingRegressor(n_estimators=100).fit(train_x, train_y)
model = SelectFromModel(clf, prefit=True)

weight = model.transform(train_x).reshape(-1)
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs
0,7.0,0.0,7.0,0.0,1.764516,6.876801,0.0,0.0,6.0,0.0,...,0.0,4.0,2.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0
1,2.0,0.0,1.0,0.0,2.000000,8.000000,6.0,0.0,6.0,1.0,...,0.0,3.0,1.0,1.0,8.0,0.0,2.0,2.0,10.0,1.0
2,7.0,0.0,7.0,0.0,2.000000,8.000000,0.0,0.0,6.0,0.0,...,0.0,3.0,1.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0
3,0.0,0.0,1.0,0.0,2.000000,8.000000,0.0,0.0,6.0,0.0,...,2.0,3.0,2.0,1.0,8.0,0.0,2.0,0.0,10.0,0.0
4,0.0,0.0,7.0,0.0,2.000000,8.000000,0.0,0.0,6.0,1.0,...,0.0,3.0,1.0,0.0,8.0,0.0,2.0,0.0,10.0,0.0


In [7]:
lgbm_hyperparams={ 'colsample_bytree': 0.3394676841949491,
                   'drop_rate': 0.005229247069083676,
                   'learning_rate': 0.08939376710901481,
                   'max_bin': 3192,
                   'max_depth': 5655,
                   'max_drop': 3654,
                   'min_child_samples': 9011,
                   'min_data_in_leaf': 1320,
                   'n_estimators': 5619,
                   'num_leaves': 8002,
                   'objective': 'regression_l1',
                   'reg_alpha': 0.3092258349709437,
                   'reg_lambda': 0.5130123398875309,
                   'skip_drop': 0.5672900003846416,
                   'verbosity': -1}

XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}



lightgbm = AdaBoostRegressor(LGBMRegressor(**lgbm_hyperparams),
                             random_state=42, 
                             n_estimators=4,
                             learning_rate=0.1)

xgboost = AdaBoostRegressor(XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                                         learning_rate=XGB_hyperparams['learning_rate'],
                                         colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                                         subsample=XGB_hyperparams['subsample'], 
                                         objective='binary:logistic',
                                         min_child_weight=XGB_hyperparams['min_child_weight'],
                                         gamma=XGB_hyperparams['gamma'],
                                         max_depth=int(XGB_hyperparams['max_depth']),
                                         reg_alpha=XGB_hyperparams['reg_alpha'],
                                         reg_lambda=XGB_hyperparams['reg_lambda']),
                            random_state=42, 
                            n_estimators=4,
                            learning_rate=0.1)

catboost = AdaBoostRegressor(CatBoostRegressor(learning_rate=0.009,
                                               depth=5,
                                               l2_leaf_reg=3.5,
                                               min_child_samples=32,
                                               grow_policy='Depthwise',
                                               iterations=100000,
                                               eval_metric='RMSE',
                                               od_type='Iter',
                                               od_wait=50,
                                               random_state=42,
                                               logging_level='Silent'),
                            random_state=42, 
                            n_estimators=4,
                            learning_rate=0.1)
#reg:logistic
#binary:logitraw
#binary:logistic

In [8]:
model = StackingRegressor(estimators=[('xgboost', xgboost), ('lightgbm', lightgbm), ('catboost', catboost)])

In [9]:
model.fit(train_x, train_y, sample_weight=weight[0:len(train_x)])

StackingRegressor(estimators=[('xgboost',
                               AdaBoostRegressor(estimator=XGBRegressor(base_score=None,
                                                                        booster=None,
                                                                        callbacks=None,
                                                                        colsample_bylevel=None,
                                                                        colsample_bynode=None,
                                                                        colsample_bytree=0.6679438268550072,
                                                                        device=None,
                                                                        early_stopping_rounds=None,
                                                                        enable_categorical=False,
                                                                        eval_metric=None,
                                                                        feature_types=None,
                                                                        gamma=2,
                                                                        grow_policy=None,
                                                                        importance_type=No...
                                                                         n_estimators=5619,
                                                                         num_leaves=8002,
                                                                         objective='regression_l1',
                                                                         reg_alpha=0.3092258349709437,
                                                                         reg_lambda=0.5130123398875309,
                                                                         skip_drop=0.5672900003846416,
                                                                         verbosity=-1),
                                                 learning_rate=0.1,
                                                 n_estimators=4,
                                                 random_state=42)),
                              ('catboost',
                               AdaBoostRegressor(estimator=<catboost.core.CatBoostRegressor object at 0x7b109796e830>,
                                                 learning_rate=0.1,
                                                 n_estimators=4,
                                                 random_state=42))])

In [10]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [11]:
test_predictions = model.predict(test)

In [12]:
0.4312733828771376

0.4312733828771376

In [13]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.325020
1,28801,0.630455
2,28802,0.191441


In [14]:
submission.to_csv('submission.csv', index = False)